# Amendment 32: the sine–cosine model on raw z

All sessions: **CPU + High-RAM**, notebook **saved in your Drive**, processor **AMD EPYC 7B12** (cell 5 checks), the same commit SHA in cell 1 (the Amendment 32 commit).

| Session | ROLE | What it does | Cells |
|---|---|---|---|
| 1 | `K1` | tunes the Fourier head on raw z (360 runs), then runs it on raw z at that rate (800 runs) | 1–7, then 8, 9 |
| 2 | `K2` | the Fourier head WITH the squash on CPU (800 runs) | 1–7, then 10 |
| any, after K1 and K2 | same | the analysis (B = 2000, about 1.5 h) | 11 |
| any, after cell 11 | `L` | re-executes every Amendment 32 run | 1–7, then 12 |
| any, after L **and after Amendment 31's J1** | `M` | the clipped runs without clipping, step check, views, both analyses without clipping | 1–7, then 13 |

**Rules:** every cell must end with `OK`; if not, stop and send the output. A Drive `Input/output error` is a Drive glitch: restart the session, run cells 1–7, then the cell again; finished runs are skipped.

In [ ]:
# 1. SETTINGS. Edit these two lines only.
EXPECTED_SHA = "PASTE_THE_FULL_SHA_OF_THE_AMENDMENT_32_COMMIT"
ROLE = "K1"            # "K1", "K2", "L" or "M"

BACKUP = "/content/drive/MyDrive/QML-backup-run"
EXP31  = BACKUP + "/exp31"                 # Amendment 26 records (read only)
OUT    = BACKUP + "/cpu_reruns"            # all CPU-check outputs; this notebook writes 35/36/37 here
REPO, ART, ENV = "/content/repo", "/content/art", "/content/env"
PY = ENV + "/bin/python"
DATASETS = ["breastmnist", "pneumoniamnist", "bloodmnist", "pathmnist"]
REGIMES  = ["5", "10", "20", "50", "100"]
TUNE_SEEDS = [90001, 90002, 90003, 90004, 90005]
T35 = T36 = 2                              # threads per process: tuning and raw-z runs (circuits benefit)
T37 = 1                                    # threads per process: the classical rate-gap runs (many processes)
EXPECT = {"38_lr_fourier_nosquash": 360, "39_fourier_cpu": 1600}

import os, subprocess, json, glob, time, shutil, re
assert ROLE in ("K1", "K2", "L", "M"), ROLE
os.environ.update({"QML_ARTIFACT_ROOT": ART, "MPLBACKEND": "Agg", "CUDA_VISIBLE_DEVICES": ""})

def sh(cmd, log=None, check=True, env=None):
    """Run a bash command from the repo, print its output, append it to a log on Drive."""
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(["bash", "-lc", cmd], cwd=REPO if os.path.isdir(REPO) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=e)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    rc = p.wait()
    if log:
        os.makedirs(OUT + "/logs", exist_ok=True)
        with open(f"{OUT}/logs/{log}", "a") as f:
            f.write(f"\n$ {cmd}\n" + "".join(lines) + f"exit code {rc}\n")
    if check and rc != 0:
        raise RuntimeError(f"FAILED (exit {rc}): {cmd}")
    return "".join(lines)

def launch(cmds, threads, tag):
    """Start commands side by side in the background, each with `threads` threads. Logs on Drive."""
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    procs = {}
    for name, cmd in cmds.items():
        log = f"{OUT}/logs/{tag}_{name}.txt"
        procs[name] = subprocess.Popen(["bash", "-lc", f"{cmd} >> {log} 2>&1"], cwd=REPO,
                                       env=env, start_new_session=True)
        print(f"started {name} (pid {procs[name].pid}, {threads} threads), log: {log}")
    return procs

def count(ns, **where):
    """Shards of a namespace on Drive whose keys match `where` (values compared as text)."""
    n = 0
    for fn in glob.glob(f"{OUT}/shards/{ns}/*.json"):
        if where:
            k = json.load(open(fn))["keys"]
            if any(str(k.get(a)) != str(v) for a, v in where.items()):
                continue
        n += 1
    return n

def run_and_watch(cmds, threads, tag, ns, total, where=None, every=900):
    """Launch, then every 15 minutes: runs done and a time estimate from the rate so far. No results."""
    where = where or {}
    n0, t0 = count(ns, **where), time.time()
    procs = launch(cmds, threads, tag)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every)
        n, dt = count(ns, **where), (time.time() - t0) / 3600
        eta = f", about {dt * (total - n) / (n - n0):.1f} h left at the rate so far" if n > n0 else ""
        print(time.strftime("%H:%M"), f"{ns}: {n} of {total}{eta}", flush=True)
    codes = {nm: p.returncode for nm, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive"
    n = count(ns, **where)
    assert n == total, f"{ns}: {n} of {total} runs"

def verdicts(target):
    """Count replay verdicts in a target namespace on Drive."""
    from collections import Counter
    c = Counter()
    for fn in glob.glob(f"{OUT}/shards/{target}/*.json"):
        try:
            v = json.load(open(fn)).get("replay_check", {}).get("verdict", "?")
        except Exception:
            v = "unreadable"
        c[v.split(" (")[0]] += 1
    return dict(c)

def halves(seq, k):
    return [seq[i::k] for i in range(k)]

def run01(arm, seeds, ns, lr, extra=""):
    """src/01 at one rate, d = 4, every dataset and n, the given seeds."""
    q = f" --lr-quantum {lr}" if arm.startswith("quantum") else ""
    return (f"{PY} -u src/01_frozen_backbone_ablation.py --arms {arm} --dims 4 "
            f"--datasets {' '.join(DATASETS)} --regimes {' '.join(REGIMES)} "
            f"--seeds {' '.join(map(str, seeds))} --lr-head {lr}{q} --experiment {ns} {extra}")

def fourier_rate():
    """Amendment 32 stage-1 selection (written by cell 8)."""
    r = json.load(open(f"{OUT}/fourier_nosquash_rate.json"))["fourier_rff"]
    assert not r["edge"], "the selection is on the grid edge: stop and send the output"
    return r["selected"]

def recorded_sha(ns):
    """The one commit that recorded a namespace (re-execution must run that commit's code)."""
    shas = {json.load(open(f))["git_sha"][:8] for f in glob.glob(f"{OUT}/shards/{ns}/*.json")}
    assert len(shas) == 1, f"{ns}: commits {shas}"
    return shas.pop()

print("ROLE", ROLE, "OK")

In [ ]:
# 2. MOUNT DRIVE and check the inputs exist.
from google.colab import drive
drive.mount("/content/drive")
fc = sorted(glob.glob(BACKUP + "/feature_cache_*.tgz"))
assert fc, "feature_cache_*.tgz not found"
n31 = len(glob.glob(EXP31 + "/shards/31_entangling_range/*.json"))
n32 = len(glob.glob(OUT + "/shards/32_squash/*.json"))
ntc = len(glob.glob(OUT + "/tune_cache/*.pt"))
print(f"31_entangling_range {n31} | 32_squash {n32} | tuning cache files on Drive {ntc}")
assert n31 == 2400 and n32 == 1600, "inputs incomplete: send this output"
print("OK")

In [ ]:
# 3. CLONE THE REPO at the pushed commit; check it is clean and carries Amendment 31.
if not os.path.isdir(REPO):
    sh(f"git clone -b feature/journal-expansion https://github.com/anishfelixm/quantum-classical-expressivity {REPO}")
sh(f"git fetch origin && git checkout --detach {EXPECTED_SHA}")
assert sh("git rev-parse HEAD").strip() == EXPECTED_SHA
assert sh("git status --porcelain").strip() == "", "the clone has local changes"
assert "## Amendment 32" in open(REPO + "/docs/analysis_plan.md").read(), "Amendment 32 missing at this commit"
for f in ("scripts/analyse_amendment32.py", "scripts/analyse_amendment31.py"):
    assert os.path.exists(f"{REPO}/{f}"), f
assert "--unbounded-angles" in open(REPO + "/src/01_frozen_backbone_ablation.py").read()
assert "--no-tanh" in open(REPO + "/src/09_lr_selection.py").read()
print("OK")

In [ ]:
# 4. PYTHON ENVIRONMENT: Python 3.10 with requirements.lock.txt (about 5 minutes).
if not os.path.exists(PY):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba")
    sh(f"/content/bin/micromamba create -y -q -p {ENV} -c conda-forge python=3.10 pip")
    sh(f"{PY} -m pip install -q -r requirements.lock.txt pytest --extra-index-url https://download.pytorch.org/whl/cu118")
sh("apt-get -qq install -y pigz > /dev/null", check=False)
sh(f"{PY} -c \"import sys, torch; from importlib.metadata import version as v; print('python', sys.version.split()[0], '| torch', torch.__version__, '| pennylane', v('pennylane'), '| numpy', v('numpy'), '| default threads', torch.get_num_threads())\"",
   log="environment_a32.txt")
print("OK")

In [ ]:
# 5. DEVICE. CPU only; record this session (cores, processor, role).
info = sh(f"{PY} -c \"import sys; sys.path.insert(0,'src'); import config; print(config.DEVICE.type)\"").split()
assert info[-1] == "cpu", "Use the CPU runtime (Runtime -> Change runtime type -> CPU, High-RAM)."
cpu = sh("lscpu | grep 'Model name' | sed 's/.*: *//'", check=False).strip()
cores = os.cpu_count()
with open(OUT + "/logs/sessions.txt", "a") as f:
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')} role={ROLE} cpu='{cpu}' cores={cores} sha={EXPECTED_SHA[:8]}\n")
print(f"cpu='{cpu}'  cores={cores}")
# Every earlier CPU run and re-execution used AMD EPYC 7B12; another processor rounds differently
# (found 9 November: an Intel session diverged at step 1). Stop rather than mix machines.
assert cpu == "AMD EPYC 7B12", ("wrong processor for this study: Runtime -> Disconnect and delete runtime, "
                                "connect again, run from cell 1")
print("OK")

In [ ]:
# 6. GATES: parity (24 head parameters at d = 4), circuit wiring, and the raw-z permission (Amendment 31).
sh(f"{PY} -m pytest -q tests/test_parity.py tests/test_entangling_ranges.py tests/test_unbounded_angles.py",
   log=f"gates_a32_{ROLE}.txt")
print("OK")

In [ ]:
# 7. ARTIFACT FOLDER: links to Drive and the restored feature cache.
def link(sub, ns, target_dir):
    os.makedirs(f"{ART}/{sub}", exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)
    p = f"{ART}/{sub}/{ns}"
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(target_dir, p)

for sub in ("shards", "predictions"):
    link(sub, "31_entangling_range", f"{EXP31}/{sub}/31_entangling_range")
for ns in ("38_lr_fourier_nosquash", "39_fourier_cpu", "C_38_lr_fourier_nosquash", "C_39_fourier_cpu",
           "CU_38_lr_fourier_nosquash", "CU_39_fourier_cpu", "CU_36_nosquash",
           "32_squash", "35_lr_nosquash", "36_nosquash", "37_rate_gaps",
           "C_35_lr_nosquash", "C_36_nosquash", "C_37_rate_gaps",
           "CU_35_lr_nosquash", "CU_36_nosquash", "CU_37_rate_gaps",
           "34_lr_inherited", "C_34_lr_inherited", "CU_34_lr_inherited",
           "C_31_entangling_range", "CU_31_entangling_range", "C_32_squash", "CU_32_squash"):
    for sub in ("shards", "predictions", "step_norms", "weights"):
        link(sub, ns, f"{OUT}/{sub}/{ns}")

FC = f"{ART}/feature_cache"
if not os.path.islink(FC):
    if os.path.isdir(FC):
        shutil.rmtree(FC)
    pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    if not pts:
        os.makedirs("/content/fc_extract", exist_ok=True)
        sh(f"tar -I pigz -xf '{fc[-1]}' -C /content/fc_extract")
        pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    os.symlink(os.path.dirname(pts[0]), FC)
assert os.path.realpath(FC).startswith("/content/fc_extract")
seeds_all = json.loads(sh(f"{PY} -c \"import sys, json; sys.path.insert(0,'src'); import config; "
                          f"print(json.dumps(config.CONFIRMATORY_SEEDS))\"").strip().splitlines()[-1])
missing = [(d, r, s) for d in DATASETS for r in REGIMES for s in seeds_all
           if not os.path.exists(f"{FC}/{d}__r{r}__s{s}.pt")]
assert not missing, f"{len(missing)} cache files missing; stop and send this output"

# the tuning seeds' caches: the files session C built and saved on 8 November. Never built here.
need = [f"{d}__r{r}__s{s}.pt" for d in DATASETS for r in ("5", "20", "100") for s in TUNE_SEEDS]
for f in need:
    if not os.path.exists(f"{FC}/{f}") and os.path.exists(f"{OUT}/tune_cache/{f}"):
        shutil.copy(f"{OUT}/tune_cache/{f}", f"{FC}/{f}")
miss = [f for f in need if not os.path.exists(f"{FC}/{f}")]
assert not miss, f"{len(miss)} tuning cache files missing, e.g. {miss[:3]}: send this output"
print("cache files present: 800 of 800, tuning 60 of 60")
print("OK")

In [ ]:
# 8. ROLE K1 - STAGE 1: tune the Fourier head on raw z (360 runs), then select its rate.
assert ROLE == "K1"
cmds = {f"t{i}": (f"{PY} -u src/09_lr_selection.py --experiment 38_lr_fourier_nosquash --arms fourier_rff "
                  f"--dim 4 --no-tanh --seeds {' '.join(map(str, s))}") for i, s in enumerate(halves(TUNE_SEEDS, 3))}
run_and_watch(cmds, 2, "tune38", "38_lr_fourier_nosquash", EXPECT["38_lr_fourier_nosquash"])
sh(f"{PY} -u scripts/analyse_amendment32.py select", log="stats_a32_select.txt")
shutil.copy(f"{ART}/fourier_nosquash_rate.json", f"{OUT}/fourier_nosquash_rate.json")
print("selected on raw z: fourier_rff", fourier_rate())
print("OK")

In [ ]:
# 9. ROLE K1 - STAGE 2: the Fourier head on raw z at its own rate (800 runs), 1 thread per process.
assert ROLE == "K1"
rf = fourier_rate()
cmds = {f"f{i}": run01("fourier_rff", s, "39_fourier_cpu", rf, "--no-tanh") for i, s in enumerate(halves(seeds_all, 8))}
run_and_watch(cmds, 1, "fc39_raw", "39_fourier_cpu", 800, where={"arm": "fourier_rff", "notanh": 1})
print("OK")

In [ ]:
# 10. ROLE K2 - STAGE 2: the Fourier head WITH the squash at 1e-2 on CPU (800 runs), 1 thread per process.
assert ROLE == "K2"
cmds = {f"f{i}": run01("fourier_rff", s, "39_fourier_cpu", "1e-2") for i, s in enumerate(halves(seeds_all, 8))}
def sq_done():
    return sum(1 for fn in glob.glob(f"{OUT}/shards/39_fourier_cpu/*.json")
               if "notanh" not in json.load(open(fn))["keys"])
procs = launch(cmds, 1, "fc39_sq")
t0, n0 = time.time(), sq_done()
while any(p.poll() is None for p in procs.values()):
    time.sleep(900)
    n, dt = sq_done(), (time.time() - t0) / 3600
    print(time.strftime("%H:%M"), f"fourier_rff squashed: {n} of 800" +
          (f", about {dt * (800 - n) / (n - n0):.1f} h left" if n > n0 else ""), flush=True)
codes = {nm: p.returncode for nm, p in procs.items()}
print("exit codes:", codes)
assert all(c == 0 for c in codes.values()) and sq_done() == 800, "send this output"
print("OK")

In [ ]:
# 11. ANY SESSION, after K1 (cell 9) and K2 (cell 10): integrity checks, then the analysis (about 1.5 h).
counts = {ns: count(ns) for ns in ("38_lr_fourier_nosquash", "39_fourier_cpu")}
print(counts, "expected", EXPECT)
assert counts == EXPECT, "runs incomplete"
recs = [json.load(open(f)) for f in glob.glob(f"{OUT}/shards/39_fourier_cpu/*.json")]
assert len({r["git_sha"] for r in recs}) == 1, "more than one commit in 39_fourier_cpu"
for f in ("fourier_nosquash_rate.json", "nosquash_rates.json"):
    shutil.copy(f"{OUT}/{f}", f"{ART}/{f}")
out = sh(f"{PY} -u scripts/analyse_amendment32.py contrasts --workers {max(1, cores - 1)}", log="stats_amendment32.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use. Do not use these numbers."
shutil.copy(f"{ART}/amendment32.json", f"{OUT}/amendment32.json")
print("OK")

In [ ]:
# 12. ROLE L - re-execute every Amendment 32 run on CPU, same threads as recorded (38: 2, 39: 1).
assert ROLE == "L"
for src, threads, groups in (
        ("38_lr_fourier_nosquash", 2, {f"t{i}": f"--seeds {' '.join(map(str, s))}" for i, s in enumerate(halves(TUNE_SEEDS, 3))}),
        ("39_fourier_cpu", 1, {f"f{i}": f"--seeds {' '.join(map(str, s))}" for i, s in enumerate(halves(seeds_all, 8))})):
    b = f"{PY} -u scripts/replay.py --source {src} --clip 20 --code-at {recorded_sha(src)}"
    first = list(groups.values())[0]
    sh(f"OMP_NUM_THREADS={threads} MKL_NUM_THREADS={threads} {b} --quick {first}", log=f"replay{src[:2]}_quick.txt")
    procs = launch({n: f"{b} {w}" for n, w in groups.items()}, threads, f"replay{src[:2]}_L")
    while any(p.poll() is None for p in procs.values()):
        time.sleep(600)
        print(time.strftime("%H:%M"), f"C_{src}:", verdicts("C_" + src), flush=True)
    codes = {nm: p.returncode for nm, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive (restart and re-run if Input/output error)"
for t in ("C_38_lr_fourier_nosquash", "C_39_fourier_cpu"):
    print(f"{t:26s} {verdicts(t)}")
print("Expected (Amendment 32 P4): C_38 IDENTICAL 360; C_39 IDENTICAL 1600.")
print("OK")

In [ ]:
# 13. ROLE M - without clipping: the runs in which clipping acted, again with clipping removed; then the step
#     check, the views and both analyses without clipping (about 2 h). Needs Amendment 31's J1 (CU_36) finished.
assert ROLE == "M"
for src, threads in (("38_lr_fourier_nosquash", 2), ("39_fourier_cpu", 1)):
    out = sh(f"{PY} scripts/build_unclipped_view.py --list {src} 2>/dev/null")
    names = sorted(set(re.findall(rf"{re.escape(src)}__\S*?\.json", out)))
    print(f"{src}: {len(names)} runs in which clipping acted")
    if not names:
        continue
    b = f"{PY} -u scripts/replay.py --source {src} --clip inf --code-at {recorded_sha(src)}"
    open(f"/content/cu{src[:2]}_first.txt", "w").write(names[0] + "\n")
    sh(f"OMP_NUM_THREADS={threads} MKL_NUM_THREADS={threads} {b} --from-list /content/cu{src[:2]}_first.txt",
       log=f"cu{src[:2]}_first.txt")
    k = max(1, min(len(names) - 1, cores // threads))
    chunks = {f"p{i}": c for i, c in enumerate(halves(names[1:], k)) if c}
    for nm, c in chunks.items():
        open(f"/content/cu{src[:2]}_{nm}.txt", "w").write("\n".join(c) + "\n")
    procs = launch({nm: f"{b} --from-list /content/cu{src[:2]}_{nm}.txt" for nm in chunks}, threads, f"cu{src[:2]}_M")
    while any(p.poll() is None for p in procs.values()):
        time.sleep(600)
        print(time.strftime("%H:%M"), f"CU_{src}:", verdicts("CU_" + src), flush=True)
    codes = {nm: p.returncode for nm, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive"
links = "31_entangling_range 36_nosquash 38_lr_fourier_nosquash 39_fourier_cpu"
sh(f"{PY} -u scripts/build_unclipped_view.py --check-steps {links}", log="a32_check_steps.txt")
sh(f"{PY} -u scripts/build_unclipped_view.py --build {links}", log="a32_views.txt")
for f in ("fourier_nosquash_rate.json", "nosquash_rates.json", "amendment32.json"):
    shutil.copy(f"{OUT}/{f}", f"{ART}/{f}")
sh(f"{PY} -u scripts/analyse_amendment32.py select --unclipped", log="stats_a32_select_unclipped.txt")
out = sh(f"{PY} -u scripts/analyse_amendment32.py contrasts --unclipped --workers {max(1, cores - 1)}",
         log="stats_amendment32_unclipped.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use."
for f in ("fourier_nosquash_rate_unclipped.json", "amendment32_unclipped.json"):
    shutil.copy(f"{ART}/{f}", f"{OUT}/{f}")
print("OK")